# Colab: SFT Training + Smoke Eval

Run these cells top to bottom on a GPU runtime. This notebook clones the latest repo, installs dependencies, trains the SFT adapter, and runs the locked-distribution smoke test.


In [ ]:
# 1. Fresh clone + dependencies.
%cd /content
!rm -rf /content/ml_project
!git clone https://github.com/Andrii238/ml-project.git /content/ml_project
%cd /content/ml_project
!pip install -q -r requirements-colab.txt
!pip uninstall -y -q torchao torchvision bitsandbytes

import torch
print('cuda:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
import transformers, trl, peft
print('transformers', transformers.__version__, '| trl', trl.__version__, '| peft', peft.__version__)


In [ ]:
# 2. Train SFT from scratch on the current locked clustered-chest dataset.
!rm -rf ckpts/sft
!PYTHONPATH=/content/ml_project PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
  python -m training.train_sft --output-dir ./ckpts/sft --num-train-epochs 3


In [ ]:
# 3. Smoke eval: quick check on the same locked clustered-chest task distribution.
!PYTHONPATH=/content/ml_project python notebooks/eval_sft_smoke.py


In [ ]:
# 4. Optional: show saved result JSON.
import json, os
path = 'results/eval_sft_vs_base_smoke.json'
print('exists:', os.path.exists(path), path)
if os.path.exists(path):
    print(json.dumps(json.load(open(path)), indent=2))
